# Python UAI324 s řešením

In [7]:
squares_even = [x**2 for x in range(1, 21) if x % 2 == 0]
print(squares_even)

[4, 16, 36, 64, 100, 144, 196, 256, 324, 400]


**Vysvětlení:** Comprehension kombinuje `for` cyklus a podmínku `if` na jednom řádku. Čitelnější a rychlejší než klasický cyklus s `append`.


### Příklad 2: Inverze slovníku

**Zadání:** Máš slovník `{jméno: věk}`. Vytvoř slovník obrácený `{věk: [jména]}` (pozor, věky se mohou opakovat).


In [8]:
people = {"Anna": 25, "Petr": 30, "Jana": 25, "Karel": 30, "Eva": 22}

by_age = {}
for name, age in people.items():
    by_age.setdefault(age, []).append(name)

print(by_age)


{25: ['Anna', 'Jana'], 30: ['Petr', 'Karel'], 22: ['Eva']}


**Alternativa s `defaultdict`:**

In [9]:
from collections import defaultdict

by_age = defaultdict(list)
for name, age in people.items():
    by_age[age].append(name)

print(dict(by_age))

{25: ['Anna', 'Jana'], 30: ['Petr', 'Karel'], 22: ['Eva']}


**Vysvětlení:** `setdefault` je užitečné, když klíč ještě neexistuje. `defaultdict` dělá totéž elegantněji.


## 2. Generátory

### Příklad 3: Fibonacciho posloupnost jako generátor

**Zadání:** Napiš generátor prvních `n` Fibonacciho čísel.


In [10]:
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

print(list(fibonacci(10)))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]



**Vysvětlení:** `yield` pozastaví funkci a vrátí hodnotu. Při dalším volání pokračuje tam, kde skončila. Ušetří paměť u velkých sekvencí.


### Příklad 4: Nekonečný generátor s `itertools`

**Zadání:** Vytvoř nekonečný cyklus barev a vypiš prvních 7.

In [11]:
from itertools import cycle, islice

colors = cycle(["červená", "zelená", "modrá"])
for c in islice(colors, 7):
    print(c)

červená
zelená
modrá
červená
zelená
modrá
červená


**Vysvětlení:** `cycle` opakuje iterovatelný objekt donekonečna. `islice` vezme jen prvních N prvků – klasický pattern pro práci s nekonečnými generátory.


## 3. Dekorátory

### Příklad 5: Měření času běhu funkce

**Zadání:** Napiš dekorátor, který změří dobu běhu funkce.


In [29]:
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} trvalo {(end - start)*1000:.2f} ms")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(i**2 for i in range(n))

@timer
def my_add(x):    
    checks=[]
    for value in x:
        checks.append(value>3)

slow_sum(1_000_000)
my_add([i for i in range(1000000)])

slow_sum trvalo 285.46 ms
my_add trvalo 49.32 ms


**Vysvětlení:** `@wraps` zachová jméno a docstring původní funkce. `*args, **kwargs` umožní dekorovat funkci s libovolnými argumenty.


### Příklad 6: Dekorátor s parametrem (retry)

**Zadání:** Napiš dekorátor `retry(attempts)`, který zkusí funkci znovu při výjimce.


In [36]:
import time
from functools import wraps

def retry(attempts=3, delay=1):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Pokus {i+1} selhal: {e}")
                    if i < attempts - 1:
                        time.sleep(delay)
            raise RuntimeError(f"Funkce {func.__name__} selhala po {attempts} pokusech")
        return wrapper
    return decorator

@retry(attempts=3, delay=0.1)
def unstable(x=3,y=0.1):
    import random
    if random.random() < 0.7:
        raise ValueError("Náhodná chyba")
    return "OK"

print(unstable())

Pokus 1 selhal: Náhodná chyba
Pokus 2 selhal: Náhodná chyba
OK


In [ ]:
print(unstable(5,1))

mohu dát unstable parametry ??

In [ ]:
Ano, můžete, ale nejdříve je musíte přidat do její definice (def unstable(a, b):). 
Váš dekorátor @retry už je na předávání parametrů plně připravený díky použití *args, **kwargs.
Pokud spustíte váš aktuální kód, skončí chybou TypeError: unstable() takes 0 positional arguments but 2 were given, protože funkce unstable žádné parametry nepřijímá.

Jak to upravit, aby to fungovalo

Stačí upravit samotnou funkci unstable (dekorátor měnit nemusíte):
python
@retry(attempts=3, delay=0.1)
def unstable(x, y):  # <-- Zde přidáme parametry
    import random
    if random.random() < 0.7:
        raise ValueError("Náhodná chyba")
    return f"OK: součet je {x + y}"

print(unstable(5, 1))
Use code with caution.

Jak to funguje uvnitř dekorátoru?

• Když zavoláte unstable(5, 1), ve skutečnosti voláte funkci wrapper(5, 1).
• Parametry 5 a 1 se zachytí do n-tice args (jako (5, 1)). Kdybyste použili pojmenované parametry (např. unstable(x=5, y=1)), 
zachytily by se do slovníku kwargs.
• Uvnitř wrapperu pak řádek return func(*args, **kwargs) tyto parametry zase rozbalí a pošle přímo vaší původní funkci unstable.
Chcete také ukázat, jak upravit dekorátor, aby dokázal zachytávat jen konkrétní typy výjimek (např. jen ValueError nebo chyby sítě), 
nebo jak předat parametry i přes klíčová slova (kwargs)?



**Vysvětlení:** Dekorátor s parametrem = funkce vracející dekorátor. Tříúrovňová struktura: `retry` → `decorator` → `wrapper`.

## 4. Context managery

### Příklad 7: Vlastní context manager pro měření času

**Zadání:** Vytvoř context manager měřící dobu běhu bloku.


In [38]:
import time
from contextlib import contextmanager

@contextmanager
def timer_block(name="blok"):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = (time.perf_counter() - start) * 1000
        print(f"[{name}] {elapsed:.2f} ms")

with timer_block("výpočet"):
    total = sum(i**2 for i in range(1_000_000))

[blok] 297.41 ms


**Vysvětlení:** `@contextmanager` převede generátor na context manager. Kód před `yield` = `__enter__`, po `yield` = `__exit__`. `finally` zajistí úklid i při výjimce.


## 5. OOP a dunder metody

### Příklad 8: Třída Vector s operátory

**Zadání:** Implementuj třídu `Vector` podporující sčítání, násobení skalárem, rovnost a `repr`.


In [42]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, k):
        return Vector(self.x * k, self.y * k)

    def __eq__(self, other):
        return isinstance(other, Vector) and (self.x, self.y) == (other.x, other.y)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __abs__(self):
        return (self.x**2 + self.y**2) ** 0.5

v1 = Vector(1, 2)
v2 = Vector(3, 4)

v3 = Vector(22, 22)

print(v1 + v2 +v3)        # Vector(4, 6)
print(v1 * 3)         # Vector(3, 6)
print(abs(v2 + v3 + v1))        # 5.0
print(v1 == Vector(1, 2))  # True

Vector(26, 28)
Vector(3, 6)
38.2099463490856
True


**Vysvětlení:** Dunder metody (`__add__`, `__mul__`, …) umožňují přirozenou syntaxi pro vlastní objekty. Vždy implementuj `__repr__` pro debugging.


### Příklad 9: Dědičnost a `super()`

**Zadání:** Vytvoř hierarchii `Zvíře` → `Pes`, `Kočka` s polymorfním `zvuk()`.


In [14]:
class Zvire:
    def __init__(self, jmeno):
        self.jmeno = jmeno
    def zvuk(self):
        raise NotImplementedError
    def __str__(self):
        return f"{type(self).__name__} {self.jmeno}: {self.zvuk()}"

class Pes(Zvire):
    def zvuk(self): return "Haf!"

class Kocka(Zvire):
    def zvuk(self): return "Mňau!"

for z in [Pes("Rex"), Kocka("Micka")]:
    print(z)

Pes Rex: Haf!
Kocka Micka: Mňau!


**Vysvětlení:** `NotImplementedError` vynutí implementaci v potomcích. Polymorfismus – volání `zvuk()` se rozhodne až za běhu podle typu.


### Příklad 10: `@dataclass` a `@property`

**Zadání:** Vytvoř datovou třídu `Teplota` s validací v setteru.


In [15]:
from dataclasses import dataclass, field

@dataclass
class Teplota:
    stupne: float
    jednotka: str = "C"

    def __post_init__(self):
        if self.jednotka not in ("C", "F", "K"):
            raise ValueError("Neplatná jednotka")
        if self.jednotka == "K" and self.stupne < 0:
            raise ValueError("Kelvin nesmí být záporný")

t = Teplota(25.5)
print(t)  # Teplota(stupne=25.5, jednotka='C')
try:
    Teplota(-5, "K")
except ValueError as e:
    print(e)  # Kelvin nesmí být záporný

Teplota(stupne=25.5, jednotka='C')
Kelvin nesmí být záporný


**Vysvětlení:** `@dataclass` automaticky vygeneruje `__init__`, `__repr__`, `__eq__`. `__post_init__` slouží pro validaci.


## 6. Funkcionální nástroje

### Příklad 11: `functools.reduce` pro agregaci

**Zadání:** Spočítej faktoriál a najdi maximum pomocí `reduce`.


In [16]:
from functools import reduce

faktorial = reduce(lambda a, b: a * b, range(1, 11))
print(faktorial)  # 3628800

slova = ["Python", "je", "super"]
veta = reduce(lambda a, b: f"{a} {b}", slova)
print(veta)  # Python je super

3628800
Python je super


**Vysvětlení:** `reduce(func, iterable)` postupně skládá hodnoty zleva doprava. Vhodné pro komutativní agregace (součet, součin, spojení).


### Příklad 12: `functools.partial` a `lru_cache`

**Zadání:** Vytvoř funkci s pevně daným argumentem a cachuj výsledky.


In [17]:
from functools import lru_cache, partial

@lru_cache(maxsize=None)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

print(fibonacci(100))  # 354224848179261915075 (okamžitě)

def mocnina(zaklad, exponent):
    return zaklad ** exponent

na_druhou = partial(mocnina, exponent=2)
print(na_druhou(7))  # 49

354224848179261915075
49


**Vysvětlení:** `lru_cache` memoizuje – bez něj by `fib(100)` běžel roky. `partial` předvyplní argumenty a vrátí novou funkci.


In [ ]:
Tento kód používá dekorátor @lru_cache (z anglického Least Recently Used Cache – mezipaměť), který funguje jako tahák pro funkci. 
    Díky němu Python nemusí počítat stejné věci pořád dokola, ale výsledky si pamatuje.
(Poznámka: partial v ukázce importujete, ale v kódu ho zatím nikde nepoužíváte.)

Proč je to bez @lru_cache pomalé?

Fibonacciho posloupnost se počítá rekurzivně (funkce volá sama sebe): fibonacci(5) potřebuje fibonacci(4) a fibonacci(3).
• Bez mezipaměti se strom výpočtů extrémně větví. Například fibonacci(3) se pro n=100 počítá miliardkrát znovu od začátku.
• Výpočet fibonacci(100) bez @lru_cache by běžnému počítači trval miliony let.

Jak to zrychlí @lru_cache(maxsize=None)?

Když nad funkci přidáte @lru_cache, Python si na pozadí vytvoří slovník (tabulku výsledků):
1. Zavoláte fibonacci(100). Funkce začne rozkládat výpočet až dolů k fibonacci(2), fibonacci(1) a fibonacci(0).
2. Jakmile poprvé spočítá např. fibonacci(3) = 2, zapíše si do paměti: "Pro vstup 3 je výsledek 2."
3. Když jiná větev výpočtu za zlomek sekundy znovu potřebuje fibonacci(3), funkce už nic nepočítá – rovnou sáhne do paměti a okamžitě vrátí 2.
4. maxsize=None říká Pythonu, že paměť pro výsledky nemá žádný limit a může si zapamatovat úplně všechny mezivýsledky.
Výsledkem je, že místo bilionů operací proběhne výpočet pro každé číslo od 0 do 100 jen jednou (celkem 101 kroků) 
a odpověď máte na obrazovce za zlomek milisekundy.

Kdy @lru_cache použít (a kdy ne)?

• Hodí se: Pro čisté funkce, které pro stejný vstup vrátí vždy stejný výsledek a výpočet trvá dlouho (matematika, složité dotazy do databáze).
• Nehodí se: Pro funkce, které pracují s náhodou (random), aktuálním časem nebo mění vnější stav (zápis do souboru), 
    protože by vám vracely zastaralý výsledek z paměti.


In [ ]:
Funkce partial (částečná aplikace funkce) vezme existující funkci, předvyplní jí některé parametry a vrátí novou, jednodušší funkci.
Funguje to jako šablona – nastavíte si výchozí hodnoty, abyste je nemuseli psát pořád dokola.

Jednoduchý příklad

Představte si obecnou funkci pro mocnění power(base, exp):
'''python
from functools import partial

def power(base, exp):
    return base ** exp

# Vytvoříme novou funkci, kde je 'exp' pevně nastaveno na 2
square = partial(power, exp=2)

# Vytvoříme další funkci pro třetí mocninu
cube = partial(power, exp=3)

print(square(5))  # 25 (zavolá power(5, exp=2))
print(cube(5))    # 125 (zavolá power(5, exp=3))
'''


K čemu se to v praxi hodí?

• Zjednodušení vestavěných funkcí: Například funkce int("1010", base=2) převádí text ve dvojkové soustavě na číslo. 
    Pomocí partial si snadno vyrobíte jednoúčelovou funkci:python
z_dvojkove = partial(int, base=2)
print(z_dvojkove("1010"))  # 10
Use code with caution.
• Předávání funkcí jako argumentů (callbacky): Když po vás nějaká knihovna (např. grafické rozhraní pro tlačítka nebo funkce map()) 
chce funkci bez parametrů, ale vaše funkce parametry potřebuje, partial je do ní předem "zabalí".

Rozdíl oproti výchozím hodnotám v def

• Výchozí hodnoty v def power(base, exp=2): měníte přímo v definici původní funkce (a máte jen jednu verzi).
• Pomocí partial můžete vzít jakoukoliv cizí funkci (kterou nemůžete nebo nechcete přepisovat) a vyrobit si z ní desítky různých variant podle potřeby.
Chcete vidět praktický příklad použití partial ve spojení s funkcí map() nebo při práci s tlačítky/událostmi, nebo máte další dotaz k modulu functools?


## 7. Práce se soubory a JSON

### Příklad 13: Načtení a zápis JSON s ošetřením chyb

**Zadání:** Bezpečně načti JSON ze souboru a zapiš zpět s hezkým formátováním.


In [19]:
import json
from pathlib import Path

def nacti_json(cesta):
    p = Path(cesta)
    if not p.exists():
        return {}
    try:
        with p.open(encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        print(f"Chyba v JSON: {e}")
        return {}

def zapis_json(cesta, data):
    with open(cesta, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

data = nacti_json("data.json")
data["uzivatele"] = [{"jmeno": "Anna", "vek": 25}, {"jmeno": "Petr", "vek": 30}]
zapis_json("data.json", data)

**Vysvětlení:** `Path` z `pathlib` je moderní práce s cestami. `ensure_ascii=False` zachová českou diakritiku čitelně. Vždy ošetřuj `JSONDecodeError`.


### Příklad 14: Čtení velkého CSV po řádcích

**Zadání:** Spočítej průměr z CSV bez načtení celého souboru do paměti.


In [20]:
import csv

def prumer_sloupce(cesta, nazev):
    total = 0
    count = 0
    with open(cesta, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                total += float(row[nazev])
                count += 1
            except (ValueError, KeyError):
                continue
    return total / count if count else None

**Vysvětlení:** `DictReader` mapuje řádky na slovníky dle hlavičky. Iterace proudem nevyžaduje načíst vše do paměti.


## 8. Ošetření výjimek

### Příklad 15: Vlastní výjimka a `finally`

**Zadání:** Vytvoř vlastní výjimku pro bankovní operaci.

In [21]:
class NedostatekPenez(Exception):
    def __init__(self, zustatek, pozadovano):
        self.zustatek = zustatek
        self.pozadovano = pozadovano
        super().__init__(f"Chybí {pozadovano - zustatek} Kč")

def vyber(zustatek, castka):
    if castka > zustatek:
        raise NedostatekPenez(zustatek, castka)
    return zustatek - castka

try:
    novy = vyber(1000, 1500)
except NedostatekPenez as e:
    print(e)  # Chybí 500 Kč
finally:
    print("Transakce ukončena")

Chybí 500 Kč
Transakce ukončena


**Vysvětlení:** Vlastní výjimky nesou kontext (atributy) a čitelnou zprávu. `finally` proběhne vždy – ideální pro úklid.


### Příklad 16: `try/except/else` a více výjimek

**Zadání:** Převod na číslo s odlišením typů chyb.


In [22]:
def bezpecny_int(s):
    try:
        return int(s)
    except TypeError:
        print("Nelze převést None")
        return None
    except ValueError:
        print(f"'{s}' není číslo")
        return None
    else:
        print("Převod OK")  # proběhne jen bez výjimky

print(bezpecny_int("42"))

42


**Vysvětlení:** `else` se spustí jen když nenastala výjimka – čistší oddělení úspěšné cesty od chyb.


## 9. Regulární výrazy

### Příklad 17: Validace e-mailu a extrakce

**Zadání:** Ověř e-mail a vytáhni z textu všechna čísla.


In [23]:
import re

EMAIL_RE = re.compile(r"^[\w.+-]+@[\w-]+\.[\w.-]+$")

def je_email(s):
    return bool(EMAIL_RE.match(s))

print(je_email("jan.novak@firma.cz"))  # True
print(je_email("špatný@email"))        # False

text = "Objednávka č. 123, cena 4500 Kč, DPH 945 Kč"
cisla = [int(n) for n in re.findall(r"\d+", text)]
print(cisla)  # [123, 4500, 945]

# pojmenované skupiny
m = re.search(r"(?P<castka>\d+)\s*Kč", text)
print(m.group("castka"))  # 4500

True
False
[123, 4500, 945]
4500



**Vysvětlení:** Kompilace regexu (`re.compile`) šetří čas při opakovaném použití. Pojmenované skupiny `(?P<nazev>...)` zpřehlední extrakci.


## 10. Kolekce a itertools

### Příklad 18: `Counter` a `most_common`

**Zadání:** Najdi 3 nejčastější slova v textu.


In [ ]:
from collections import Counter
import re

text = """Python je super. Python je rychlý a Python je čitelný.
Programátoři milují Python, protože Python je jednoduchý."""

slova = re.findall(r"\w+", text.lower())
top3 = Counter(slova).most_common(3)
print(top3)

**Vysvětlení:** `Counter` je `dict` specializovaný na počítání. `most_common(n)` vrací seřazený seznam n nejčastějších.


### Příklad 19: Kombinace `groupby` a `sorted`

**Zadání:** Seskupte zaměstnance podle oddělení.

In [ ]:
from itertools import groupby
from operator import itemgetter

zamci = [
    ("Anna", "IT"), ("Petr", "HR"), ("Jana", "IT"),
    ("Karel", "Finance"), ("Eva", "HR"),
]

zamci.sort(key=itemgetter(1))  # groupby vyžaduje seřazení!
for oddeleni, skupina in groupby(zamci, key=itemgetter(1)):
    print(oddeleni, [j for j, _ in skupina])

**Vysvětlení:** `groupby` funguje jen na **seřazeném** vstupu – častá chyba. `itemgetter` je efektivnější než `lambda`.


## 11. Typové anotace

### Příklad 20: Typované funkce a validace za běhu

**Zadání:** Napiš typovanou funkci a ověř typy pomocí `isinstance`.


In [ ]:
from typing import Iterable, Optional

def prumer(cisla: Iterable[float]) -> Optional[float]:
    soucet = 0.0
    pocet = 0
    for c in cisla:
        if not isinstance(c, (int, float)):
            raise TypeError(f"Očekáváno číslo, dostal jsem {type(c).__name__}")
        soucet += c
        pocet += 1
    return soucet / pocet if pocet else None

print(prumer([1, 2, 3.5]))   # 2.1666666666666665
print(prumer([]))            # None

**Vysvětlení:** Typy nejsou v Pythonu vynuceny, ale slouží jako dokumentace a jsou kontrolovatelné nástroji jako `mypy`. Explicitní kontrola `isinstance` je užitečná pro veřejná API.


## Doporučení pro výuku

- **Pořadí:** 1 → 2 → 3 → 4 (nejdřív základy, pak abstrakce).
- **Cvičení k úpravám:** Nech studenty měnit zadání (např. retry s exponenciálním backoffem, `Vector` s odečítáním).
- **Diskuze:** Kdy použít generátor vs. seznam; kdy dědičnost vs. kompozice; proč `lru_cache` funguje jen pro hashovatelné argumenty.
- **Propojení:** Dekorátor `timer` jde napsat i jako context manager – ideální srovnání obou přístupů.
